# Bronze Layer Ingestion Pipeline
## Serie A Player Statistics | API-Football → Databricks Lakehouse

---

### Architecture (5-Cell Structure)

| Cell | Purpose |
|---|---|
| Cell 1 | Pipeline Configuration |
| Cell 2 | Pre-Flight Metadata Validation |
| Cell 3 | Landing Zone Initialization |
| Cell 4 | Distributed Extraction Engine |
| Cell 5 | Auto Loader Delta Commit |

**Output:** `bronze_dev.api_sports.raw_serie_a_players_2024`

> **Prerequisites:** `01_infrastructure_setup.ipynb` must have been 
> run at least once before executing this notebook.

## Cell 1: Pipeline Configuration
Modify only this cell to target a different league, country, or season.

> **API Key Security:** Store your key in a local `.env` file,
> the `.gitignore` in this repository excludes it automatically. 
> Never commit credentials to version control.

In [0]:
import requests
import time
import json
from datetime import datetime

# =============================================================================
# RUNTIME PARAMETERS
# =============================================================================
API_KEY            = "YOUR_API_KEY_HERE"
TARGET_LEAGUE_NAME = "Serie A"
TARGET_COUNTRY     = "Italy"
TARGET_SEASON      = "2024"

HEADERS     = {"x-apisports-key": API_KEY}
URL_LEAGUES = "https://v3.football.api-sports.io/leagues"
URL_TEAMS   = "https://v3.football.api-sports.io/teams"
URL_PLAYERS = "https://v3.football.api-sports.io/players"

print(f"Pipeline configured for: {TARGET_LEAGUE_NAME}, {TARGET_COUNTRY} ({TARGET_SEASON})")

## Cell 2: Pre-Flight Metadata Validation
Validates all runtime parameters against the source API before 
consuming pagination quota. Raises `ValueError` immediately on 
invalid parameters.

**API calls consumed:** 2

In [0]:
print("Validating parameters with source system...")

# Omits "current": "true" — allows historical season access
league_params = {"name": TARGET_LEAGUE_NAME, "country": TARGET_COUNTRY}
league_response = requests.get(
    URL_LEAGUES, headers=HEADERS, params=league_params
).json()

# Empty response indicates invalid league name or country
if not league_response.get("response"):
    raise ValueError(
        f"Could not find '{TARGET_LEAGUE_NAME}' in '{TARGET_COUNTRY}'. "
        f"Check spelling in Cell 1."
    )

league_id = league_response["response"][0]["league"]["id"]
print(f"League validated. Internal ID: {league_id}")

# Team IDs become extraction targets for Cell 4
team_params = {"league": league_id, "season": TARGET_SEASON}
teams_response = requests.get(
    URL_TEAMS, headers=HEADERS, params=team_params
).json()
team_ids = [item["team"]["id"] for item in teams_response.get("response", [])]

# Empty team list indicates invalid or future season
if not team_ids:
    raise ValueError(
        f"No teams found for the {TARGET_SEASON} season. "
        f"Verify the year in Cell 1."
    )

print(f"Season validated. Located {len(team_ids)} teams.")
print(f"Team IDs: {team_ids}")

## Cell 3: Landing Zone Initialization
Defines path variables and ensures the run folder exists in the 
Unity Catalog Volume.

> Catalog, schema, and volume provisioning is handled by 
> `01_infrastructure_setup.ipynb` — this cell only creates the 
> run-specific folder within the existing Volume.

In [0]:
clean_league_name = TARGET_LEAGUE_NAME.lower().replace(" ", "_")

catalog_name = "bronze_dev"
schema_name  = "api_sports"
table_name   = f"raw_{clean_league_name}_players_{TARGET_SEASON}"
volume_name  = "landing_zone"

volume_folder_path = (
    f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
    f"/{clean_league_name}_{TARGET_SEASON}/"
)
checkpoint_path = (
    f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
    f"/_checkpoints/{clean_league_name}_{TARGET_SEASON}/"
)
full_table_path = f"{catalog_name}.{schema_name}.{table_name}"

# Preserves previously landed files — Auto Loader checkpoint tracks processed files
dbutils.fs.mkdirs(volume_folder_path)

print(f"Landing zone folder secured at: {volume_folder_path}")
print(f"Delta table target: {full_table_path}")

## Cell 4: Distributed Extraction Engine
Iterates through all validated Team IDs, handles dynamic pagination, 
and writes each API response as a uniquely timestamped JSON file.

**Key decisions:**
- **Timestamped filenames** — every run produces unique filenames for 
  correct Auto Loader detection
- **7-second delay** — respects the free tier 10 requests/minute limit
- **Hard limit break** — free tier caps at page 3 per team; exits 
  cleanly rather than retrying
- **Transient error retry** — 15-second pause and retry on network 
  errors or rate limit spikes

> **Expected duration:** ~8-10 minutes for 20 teams

In [0]:
# Single timestamp per run — all files from this run share the same timestamp
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Starting extraction. Run timestamp: {RUN_TIMESTAMP}")

for team_id in team_ids:
    current_page         = 1
    total_pages_for_team = 1

    while current_page <= total_pages_for_team:

        player_params = {
            "team":   team_id,
            "season": TARGET_SEASON,
            "page":   current_page
        }
        response = requests.get(
            URL_PLAYERS, headers=HEADERS, params=player_params
        ).json()

        # Hard limit vs transient error — different handling required
        if response.get("errors"):
            error_msg = str(response["errors"])
            if "Page parameter" in error_msg:
                # Hard limit — free tier page cap, no point retrying
                print(f"  Page limit reached for team {team_id} at page {current_page}. Moving on.")
                break
            else:
                # Transient error — pause and retry same page
                print(f"  Transient error on team {team_id} page {current_page}: {error_msg}. Retrying in 15s...")
                time.sleep(15)
                continue

        # Read total pages from API metadata on first page only
        if current_page == 1 and "paging" in response:
            total_pages_for_team = response["paging"]["total"]

        if response.get("response"):
            # Timestamped filename — Auto Loader always detects as new
            file_name = f"team_{team_id}_page_{current_page}_{RUN_TIMESTAMP}.json"
            file_path = f"{volume_folder_path}{file_name}"

            with open(file_path, "w") as f:
                for player_record in response["response"]:
                    f.write(json.dumps(player_record) + "\n")

            print(f"  Written: {file_name}")

        current_page += 1
        time.sleep(7)  # Rate limit guard — stays under 10 req/min

print(f"\nExtraction complete. Files landed in: {volume_folder_path}")

## Cell 5: Auto Loader Streaming & Delta Lake Commit
Incrementally ingests new JSON files into the Bronze Delta table 
using Databricks Auto Loader.

In [0]:
print("Initializing Auto Loader...")

# checkpoint: schema persistence + processed file tracking
df_raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(volume_folder_path)
)

query = (
    df_raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")  # handles additive API schema changes
    .trigger(availableNow=True)     # process all new files then shut down
    .toTable(full_table_path)
)

# Blocks until async stream fully commits to Delta
query.awaitTermination()

print(f"Auto Loader commit complete. Table: {full_table_path}")

## Bronze Layer Complete

| Property | Value |
|---|---|
| **Output Delta Table** | `bronze_dev.api_sports.raw_serie_a_players_2024` |
| **Storage** | Unity Catalog Volume → Delta Lake |
| **Compliance** | ACID transactions, schema evolution enabled |
| **Data State** | Raw, untransformed — exact API payload preserved |

> Records per run: ~1,126 players across 20 clubs (2024 season).  
> The Bronze Delta table grows incrementally with each run.

---

**Next Step — LakeFlow Transformation Pipeline**

Trigger the LakeFlow pipeline to transform Bronze through Silver 
and Gold.

> In production this step is automated via the Databricks Workflow  
> `serie_a_colombian_players_daily_pipeline`.